# v2 · 04 · Richer behaviour states and objective pattern mining
v1 described each month with **5 repayment states** and used **patterns I chose by hand** (streaks, relapses...).
Here:
1. each month gets a **richer state** = repayment group × card-usage band × how much of last month's bill was paid (60 states occur);
2. **every** 3-month pattern is tested objectively, keeping only those that are significant after correcting for testing many patterns, *and* that still matter among customers with the same number of late months.

All discovery uses **training customers only**. This notebook does the discovery and model tests in pandas; `v2_05` loads the same states and patterns into a separate v2 Neo4j graph.

**Setup.**

In [1]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))      # v2 package (adds v1 src to the path too)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from credit_kg_v2 import load_config, OUT
from credit_kg import config as C, scoring as S, modeling as M
import json
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from credit_kg import data as D
from credit_kg_v2 import rich_states as RS

**Load v1's prepared data and split.**

In [2]:
wide, monthly, split = D.load_processed()
w = wide.merge(split); tr = w[w.split == "train"]
late_tr = (tr[[f"status_m{m}" for m in range(1, 7)]] >= 1).sum(axis=1)

**The richer states.** Each month = repayment group | usage band | payment band. Here are the most common ones, and customer 1's six months.

In [3]:
st = RS.rich_states(w)
print("distinct rich states:", len(np.unique(st)))
display(pd.Series(st.ravel()).value_counts().head(12).to_frame("customer-months"))
pd.DataFrame(st[w.customer_id == 1], columns=["Apr", "May", "Jun", "Jul", "Aug", "Sep"])

distinct rich states: 60


,customer-months
REVOLVING|MID|WEAK,30985
PAID_FULL|LOW|FULL,18415
REVOLVING|HIGH|WEAK,15885
NO_USE|LOW|NODUE,10359
REVOLVING|LOW|WEAK,10120
REVOLVING|LOW|PART,8943
NO_USE|LOW|FULL,8536
REVOLVING|MID|NA,8336
LATE|MID|WEAK,7880
REVOLVING|LOW|NA,5938


,Apr,May,Jun,Jul,Aug,Sep
0,NO_USE|LOW|NA,NO_USE|LOW|NODUE,PAID_FULL|LOW|NODUE,PAID_FULL|LOW|NODUE,LATE|LOW|FULL,LATE|LOW|WEAK


**Do richer states separate risk better than v1's 5 states?** Default rate by the **September** state, training customers.
Within the same repayment group (e.g. REVOLVING), usage and payment bands split customers into quite different risk levels.

In [4]:
sep = pd.Series(RS.rich_states(tr)[:, -1], index=tr.index)
t = tr[C.TARGET].groupby(sep).agg(customers="size", default_rate="mean")
t = t[t.customers >= 150].sort_values("default_rate")
t.round(3)

,customers,default_rate
REVOLVING|LOW|WEAK,1016,0.054
NO_USE|LOW|FULL,1330,0.077
REVOLVING|MID|WEAK,3926,0.111
REVOLVING|LOW|FULL,321,0.112
REVOLVING|LOW|PART,1634,0.115
PAID_FULL|LOW|FULL,3010,0.150
PAID_FULL|LOW|PART,347,0.156
PAID_FULL|LOW|NODUE,536,0.157
REVOLVING|HIGH|WEAK,3171,0.161
REVOLVING|HIGH|PART,249,0.173


**Month-to-month movement between rich states** (training customers): the probability of being late next month for each state. Rare states are pulled towards the overall average so a handful of customers can't produce extreme numbers.

In [5]:
st_tr = RS.rich_states(tr)
late_mat = (tr[[f"status_m{m}" for m in range(1, 7)]] >= 1).to_numpy()
p_late = RS.transition_late_prob(st_tr, late_mat)
pd.Series({k: v for k, v in p_late.items() if k != "__overall__"}).sort_values(ascending=False).round(3).to_frame("P(late next month)").head(10)

,P(late next month)
LATE|HIGH|WEAK,0.851
LATE|MID|WEAK,0.828
LATE|MID|PART,0.780
LATE|LOW|WEAK,0.760
LATE|HIGH|NA,0.738
LATE|MID|NA,0.684
LATE|LOW|NODUE,0.654
LATE|HIGH|PART,0.648
LATE|LOW|PART,0.576
LATE|LOW|NA,0.562


**Objective pattern mining.** Every 3-month pattern of (repayment group / payment band) shown by at least 200 training customers is tested:
* `significant`: default rate differs from the rest, after Benjamini–Hochberg correction for testing many patterns;
* `adds_beyond_late_count`: still significant **among customers with the same number of late months** (Mantel–Haenszel test), so it tells us something the simple late count doesn't.

In [6]:
mot = RS.mine_motifs(tr, tr[C.TARGET], late_tr)
mot.to_csv(OUT / "motifs_mined_train.csv", index=False)
print(f"candidate patterns: {len(mot)} | significant: {mot.significant.sum()} | add beyond late count: {mot.adds_beyond_late_count.sum()}")
mot[mot.adds_beyond_late_count][["motif", "customers", "default_rate", "lift", "or_within_late_groups", "q_within"]].round(3)

candidate patterns: 53 | significant: 45 | add beyond late count: 11


,motif,customers,default_rate,lift,or_within_late_groups,q_within
0,REVOLVING/WEAK > REVOLVING/WEAK > LATE/WEAK,1513,0.517,2.337,1.921,0.000
1,NO_USE/NODUE > NO_USE/NODUE > NO_USE/NODUE,1438,0.273,1.235,1.706,0.000
3,NO_USE/FULL > NO_USE/FULL > NO_USE/FULL,937,0.077,0.347,0.607,0.001
4,REVOLVING/PART > REVOLVING/PART > REVOLVING/PART,1340,0.107,0.482,0.713,0.003
5,REVOLVING/WEAK > LATE/WEAK > LATE/WEAK,1270,0.517,2.335,1.268,0.012
7,PAID_FULL/FULL > PAID_FULL/FULL > NO_USE/FULL,204,0.093,0.421,0.477,0.016
8,NO_USE/NODUE > REVOLVING/NODUE > REVOLVING/NODUE,461,0.176,0.794,1.460,0.024
9,NO_USE/NODUE > NO_USE/NODUE > LATE/NODUE,713,0.360,1.629,1.280,0.024
10,LATE/WEAK > LATE/WEAK > REVOLVING/WEAK,356,0.427,1.930,0.719,0.024
13,REVOLVING/WEAK > LATE/WEAK > REVOLVING/WEAK,486,0.261,1.181,0.749,0.034


**Reading the discoveries** (odds ratio within late-count groups: >1 riskier, <1 safer than customers with the *same* number of late months):
* *Paying under 10% of the bill for two months, then going late* (odds ×1.9) is riskier than other lateness of the same count. It's a slide, not a blip.
* *No card use and nothing due for three months* (×1.7) is riskier: inactivity is not safety (matches v1's U-shape).
* But *no new spending while still clearing small bills in full* (×0.61) is **safer**: the same "inactive" label hides two very different behaviours.
* *Paying in full, then going quiet* (×0.48) and *steady partial payments while revolving* (×0.71) are safer than similar late-count profiles.

In [7]:
sel = mot[mot.adds_beyond_late_count]
motif_list = sel.motif.tolist(); risky = set(sel.loc[sel.or_within_late_groups > 1, "motif"])
print(len(risky), "risky patterns,", len(motif_list) - len(risky), "protective patterns")

5 risky patterns, 6 protective patterns


**Turn the discoveries into 5 features** for every customer: September-state late probability, average path late probability, number of distinct states, and counts of risky and protective patterns.
Note: the patterns were *selected* using training customers' outcomes, so cross-validation on training data is slightly optimistic. **The test-set comparison is the decisive one.**

In [8]:
rf_ = RS.rich_features(w, p_late, motif_list, risky)
rf_.to_parquet(OUT / "rich_state_features.parquet", index=False)
table = M.load_table().merge(rf_, on="customer_id")
RICH = [c for c in rf_.columns if c != "customer_id"]
for p in ["train", "test"]:
    t = table[table.split == p]
    print(p, {c: round(max(roc_auc_score(t[C.TARGET], t[c]), 1 - roc_auc_score(t[C.TARGET], t[c])), 3) for c in RICH})

train {'rs_next_late_prob': 0.73, 'rs_mean_late_prob': 0.752, 'rs_n_distinct_states': 0.536, 'motif_risky_count': 0.58, 'motif_protective_count': 0.52}
test {'rs_next_late_prob': 0.716, 'rs_mean_late_prob': 0.741, 'rs_n_distinct_states': 0.521, 'motif_risky_count': 0.58, 'motif_protective_count': 0.523}


## Does it improve default prediction? (v1 protocol)
A vs A + rich features, for the Random Forest and for logistic regression: same folds, OOF threshold, test set used once, paired bootstrap.

In [9]:
params = json.load(open(C.MODELS / "rf_params.json"))
def rf():
    mm = M.make_rf(params["min_samples_leaf"]); mm.set_params(max_features=params["max_features"]); return mm
lr = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
fs = M.feature_sets(); colsA = fs["A"]; colsR = colsA + RICH
tr_, te_ = table[table.split == "train"], table[table.split == "test"]
y, yt = tr_[C.TARGET].to_numpy(), te_[C.TARGET].to_numpy(); n_tr, n_te = int(len(y) * .8), int(len(y) * .2)
cvA = {"RF": pd.read_csv(C.TABLES / "cv_folds_A.csv"), "LR": pd.read_csv(C.TABLES / "cv_folds_LR-A.csv")}
rows, boots, tests = {}, {}, {}
for name, mk in [("RF", rf), ("LR", lr)]:
    cvr = M.repeated_cv(tr_[colsR], y, mk)
    tests[name] = M.corrected_resampled_ttest(cvr.roc_auc - cvA[name].roc_auc, n_tr, n_te) | {"cv_A": cvA[name].roc_auc.mean(), "cv_A+rich": cvr.roc_auc.mean()}
    preds = {}
    for label, cols_ in [("A", colsA), ("A+rich", colsR)]:
        thr = M.best_f1_threshold(y, M.oof_proba(tr_[cols_], y, mk))
        preds[label] = mk().fit(tr_[cols_], y).predict_proba(te_[cols_])[:, 1]
        rows[f"{name} {label}"] = M.evaluate(yt, preds[label], thr)
    boots[name] = M.paired_bootstrap(yt, preds["A"], preds["A+rich"])
    b = boots[name]; print(f"{name}: test ΔROC-AUC {b['delta_roc_auc']:+.4f} CI [{b['roc_auc_ci'][0]:+.4f}, {b['roc_auc_ci'][1]:+.4f}] | ΔPR-AUC {b['delta_pr_auc']:+.4f} CI [{b['pr_auc_ci'][0]:+.4f}, {b['pr_auc_ci'][1]:+.4f}]")
display(pd.DataFrame(tests).T.round(4))
res = pd.DataFrame(rows).loc[["roc_auc", "pr_auc", "precision", "recall", "f1"]]; res.round(4)

RF: test ΔROC-AUC -0.0017 CI [-0.0034, +0.0000] | ΔPR-AUC -0.0035 CI [-0.0064, -0.0005]


LR: test ΔROC-AUC +0.0030 CI [+0.0002, +0.0056] | ΔPR-AUC -0.0032 CI [-0.0092, +0.0028]


,mean_diff,folds_b_better,folds,t,p_value,cv_A,cv_A+rich
RF,-0.0004,11.0,25.0,-0.738,0.4676,0.7880,0.7876
LR,0.0042,25.0,25.0,5.766,0.0000,0.7665,0.7707


,RF A,RF A+rich,LR A,LR A+rich
roc_auc,0.7800,0.7783,0.7573,0.7602
pr_auc,0.5575,0.5540,0.5021,0.4989
precision,0.4975,0.5055,0.4827,0.4851
recall,0.5961,0.5855,0.5576,0.5539
f1,0.5423,0.5426,0.5175,0.5172


**Save** the comparison.

In [10]:
res.to_csv(OUT / "rich_states_comparison_test.csv")
with open(OUT / "rich_states_comparison.json", "w") as fh:
    json.dump({"cv_tests": tests, "bootstrap": boots, "motifs_selected": motif_list, "risky": sorted(risky)}, fh, indent=2, default=float)
print("saved")

saved


### Takeaways
* **Richer states separate risk much better.** Within "REVOLVING" alone, September default rates range from 5% (low usage, small bills) to 17% (high usage); within "LATE", from 23% to 69%.
* **Objective pattern mining found 11 patterns that add information beyond the late-month count** (out of 53 tested, 45 significant). They include a slide from weak payments into lateness (riskier), and two different kinds of "inactivity" with opposite risk.
* **Models:**
  - Random Forest: no gain (test ΔROC-AUC −0.0017, CI −0.0034 to 0.0000);
  - **logistic regression: a small reliable gain** (better in all 25 folds; test ΔROC-AUC **+0.0030, 95% CI +0.0002 to +0.0056**), the same pattern as v1's graph features;
  - PR-AUC does not improve for either model.
* **Value:** mainly **discovery and explanation** (patterns nobody hand-picked, each with an odds ratio), plus a small gain for the explainable model. The accuracy ceiling stays where the data puts it.